# L22 · Policy learning con presupuesto

**S22 · Lectura: Cap. 21 · 40–55 min**

## La pregunta

¿A quién tratar si solo podemos atender al 20 %?

## Qué harás

Construir y comparar una política causal bajo presupuesto.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Policy top-k, valor y comparación con baselines.

> **Idea clave:** El mejor ranking de riesgo no siempre es la mejor política de intervención.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import LogisticRegression
rng = setup()
sc = load_scenario("real/ihdp"); sc.card()
cols = list(sc.covariates)
Xtr, Ytr, Ttr = sc.xyt("train"); Ttr = Ttr.astype(int)
Xv, Yv, Tv = sc.xyt("val"); Xt, Yt, Tt = sc.xyt("test")
XH = np.vstack([Xv, Xt]); YH = np.r_[Yv, Yt]; TH = np.r_[Tv, Tt].astype(int)          # HOLD-OUT = validación + prueba
print("entrenamiento:", len(Ytr), "· hold-out (validación + prueba):", len(YH))

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

## 1. Del CATE a una política con presupuesto (§21.2, §21.3)
Valor de una política: $V(\pi)=E[Y(\pi(X))]=E[\mu_0(X)+\pi(X)\tau(X)]$. La política óptima sin restricciones es $\mathbf 1\{\tau(x)>0\}$. Con un presupuesto $\beta$ (aquí tratar a lo sumo el **30 %**), se trata a quienes tienen el $\hat\tau$ más alto.

Primero estimamos nuisance y CATE **solo con el conjunto de entrenamiento**.

In [ ]:
ETR = lambda: ExtraTreesRegressor(n_estimators=40, min_samples_leaf=8, max_features=0.8, random_state=0)
def psi_cf(X, y, d, K=2, seed=SEED):
    n = len(y); psi = np.zeros(n)
    for te in kfold(n, K, seed):
        tr = np.setdiff1d(np.arange(n), te); t1, t0 = tr[d[tr] == 1], tr[d[tr] == 0]
        m1 = ETR().fit(X[t1], y[t1]).predict(X[te]); m0 = ETR().fit(X[t0], y[t0]).predict(X[te])
        e = np.clip(LogisticRegression(max_iter=500).fit(X[tr], d[tr]).predict_proba(X[te])[:, 1], .05, .95)
        psi[te] = m1 - m0 + d[te] * (y[te] - m1) / e - (1 - d[te]) * (y[te] - m0) / (1 - e)
    return psi
psi_tr = psi_cf(Xtr, Ytr, Ttr)
cate_model = ETR().fit(Xtr, psi_tr)                                    # DR-learner: regresión de las puntuaciones AIPW
tau_tr, tau_H = cate_model.predict(Xtr), cate_model.predict(XH)
# nuisance para evaluar en hold-out (entrenados SOLO con train)
mu1_m = ETR().fit(Xtr[Ttr == 1], Ytr[Ttr == 1]); mu0_m = ETR().fit(Xtr[Ttr == 0], Ytr[Ttr == 0])
e_m = LogisticRegression(max_iter=500).fit(Xtr, Ttr)
mu1_H, mu0_H = mu1_m.predict(XH), mu0_m.predict(XH); e_H = np.clip(e_m.predict_proba(XH)[:, 1], .05, .95)
BETA = 0.30
def top_frac(score, beta=BETA):
    k = int(np.floor(beta * len(score))); pi = np.zeros(len(score), int); pi[np.argsort(-score)[:k]] = 1; return pi
pi_top = top_frac(tau_H)
print(f"política top-{int(BETA*100)} % por τ̂: trata al {100 * pi_top.mean():.1f} % del hold-out")

## 2. Árbol de política (profundidad 3) y presupuesto (Alg. 9)
Se construye un árbol de profundidad 3 con **objetivo $\hat\tau$** y se trata a las hojas cuyo $\hat\tau$ medio supera el umbral $c$ = cuantil 70 % de los $\hat\tau$ de entrenamiento. Esto produce **reglas legibles**, a costa de aproximar la regla óptima.

In [ ]:
arbol = DecisionTreeRegressor(max_depth=3, min_samples_leaf=25, random_state=0).fit(Xtr, tau_tr)
c = np.quantile(tau_tr, 1 - BETA)
trata_hoja = {h: arbol.tree_.value[h][0][0] > c for h in range(arbol.tree_.node_count) if arbol.tree_.children_left[h] == -1}
pi_tree = np.array([trata_hoja[h] for h in arbol.apply(XH)]).astype(int)
print(export_text(arbol, feature_names=cols, max_depth=3))
print(f"fracción tratada por el árbol en hold-out: {100 * pi_tree.mean():.1f} %  (presupuesto: {int(BETA * 100)} %)")

## 3. Evaluar **con** y **sin** la verdad
En simulación (IHDP es semisintético) conocemos $\tau_i$ y podemos medir el *welfare* oráculo $E[\pi\tau]$. Con datos reales **no existe**: solo se puede **estimar** el valor de la política con los datos observados.

Tres estimadores de $V(\pi)$ sobre el hold-out ($e_\pi=e$ si $\pi=1$, $1-e$ si $\pi=0$):
- **IPS:** $\frac1n\sum \mathbf 1\{T_i=\pi_i\}Y_i/e_\pi$;
- **SNIPS:** IPS normalizado por la suma de pesos;
- **Doblemente robusto (DR):** $\frac1n\sum[\hat\mu_\pi+\mathbf 1\{T=\pi\}(Y-\hat\mu_\pi)/e_\pi]$.

### Completa tú
`valor_politica_dr(pi, y, t, mu0, mu1, e)` → `(V, se, psi)` con `psi` el vector de puntuaciones (su desviación da el error estándar por función de influencia).

In [ ]:
def valor_ips(pi, y, t, e):
    e_pi = np.where(pi == 1, e, 1 - e); return float(np.mean((t == pi) * y / e_pi))
def valor_snips(pi, y, t, e):
    e_pi = np.where(pi == 1, e, 1 - e); w = (t == pi) / e_pi; return float((w * y).sum() / w.sum())

In [ ]:
# ✏️ Completa tú — valor_politica_dr
# Pista: mu_pi = mu1 donde pi==1 y mu0 donde pi==0; e_pi = e donde pi==1 y 1-e donde pi==0; psi = mu_pi + 1{t==pi}/e_pi * (y - mu_pi).
def valor_politica_dr(pi, y, t, mu0, mu1, e):
    """Valor DR de la política pi. Devuelve (V, se, psi)."""
    raise NotImplementedError("completa valor_politica_dr")

In [ ]:
# caso de verificación: mu = 0, e = 0,5, pi = t  →  psi_i = y_i / 0,5
y_, t_ = np.array([1., 2.]), np.array([1, 0])
lab_check("valor_politica_dr", lambda: (valor_politica_dr(np.array([1, 0]), y_, t_, np.zeros(2), np.zeros(2), np.full(2, .5))[0], valor_politica_dr(np.array([1, 1]), y_, t_, np.zeros(2), np.zeros(2), np.full(2, .5))[0]),
          [((), (3.0, 1.0), 1e-9)])

In [ ]:
# EVALUATION-CELL (usa la verdad simulada)
def todo_pol(pi):
    V, se, psi = valor_politica_dr(pi, YH, TH, mu0_H, mu1_H, e_H); return V, se, psi
V0, se0, psi0 = todo_pol(np.zeros(len(YH), int))                      # no tratar a nadie
tauH_true = None if sc.tau("val") is None else np.r_[sc.tau("val"), sc.tau("test")]
filas = {}
for nom, pi in (("tratar a todos", np.ones(len(YH), int)), ("top-30 % por τ̂", pi_top), ("árbol de política", pi_tree)):
    V, se, psi = todo_pol(pi); inc = psi - psi0; ips = valor_ips(pi, YH, TH, e_H) - valor_ips(np.zeros(len(YH), int), YH, TH, e_H); sn = valor_snips(pi, YH, TH, e_H) - valor_snips(np.zeros(len(YH), int), YH, TH, e_H)
    f = {"fracción tratada": pi.mean(), "IPS (ganancia)": ips, "SNIPS (ganancia)": sn, "DR (ganancia)": inc.mean(), "se DR": inc.std(ddof=1) / np.sqrt(len(inc))}
    if tauH_true is not None: f["oráculo E[π·τ] (EVALUATION)"] = float(np.mean(pi * tauH_true))
    filas[nom] = f
show(pd.DataFrame(filas).T.round(3))

**Mira el resultado.** Compara las columnas DR y oráculo, y mira el error estándar. ¿Qué te dice el tamaño del IC con ~300 filas de hold-out?

**Respuesta:**

El DR se acerca al oráculo pero con un **error estándar grande**: con pocas filas de evaluación, una diferencia entre políticas puede ser indistinguible de cero. IPS suele ser más ruidoso (pesos grandes) y SNIPS lo estabiliza a cambio de un pequeño sesgo. El oráculo es un lujo de la simulación: con datos reales solo existe la columna DR (o IPS/SNIPS) y su incertidumbre.



## 4. Señal de alerta: evaluar dentro de muestra, o el Listado 21.2 circular (E6)
El Listado 21.2 calcula `welfare_budget = cate[pi==1].mean() * pi.mean()` con **el mismo CATE estimado que eligió a quién tratar**: seleccionar a los de mayor $\hat\tau$ y luego promediar $\hat\tau$ entre ellos es optimista por construcción (maldición del ganador). Comparamos tres cifras para la política top-30 %.

In [ ]:
circular_tr = tau_tr[top_frac(tau_tr) == 1].mean() * 0.30                      # lo que haría el Listado 21.2 (en train, con el mismo τ̂)
circular_H = tau_H[pi_top == 1].mean() * pi_top.mean()                         # idem sobre el hold-out con el τ̂ que elige
dr_H = (psi_pol := todo_pol(pi_top))[0] - V0
filas = {"circular (Listado 21.2, en train)": circular_tr, "circular (mismo τ̂ elige y evalúa, hold-out)": circular_H, "DR en hold-out (honesto)": dr_H}
# EVALUATION-CELL
if tauH_true is not None: filas["oráculo en hold-out (verdad)"] = float(np.mean(pi_top * tauH_true))
if sc.tau("train") is not None: filas["oráculo en train, misma política (verdad)"] = float(np.mean(top_frac(tau_tr) * sc.tau("train")))
show(pd.Series(filas, name="welfare incremental de la política top-30 %").round(3).to_frame())

**Mira el resultado.** ¿Qué cifra es la más alta, cuál es más cercana a la verdad y por qué la circular es optimista?

**Respuesta:**

Compara cada cifra con el oráculo de **su misma muestra**: la circular en train queda por encima del oráculo de train (el mismo $\hat\tau$, con su ruido, selecciona y mide, así que se queda con el ruido favorable); cuanto más flexible el CATE, mayor la brecha. La DR en hold-out no usa el $\hat\tau$ para medir, sino resultados observados de unidades no vistas por el modelo: es la que se acerca a la verdad del hold-out (dentro de su incertidumbre). Regla: **el welfare se reporta en una muestra de evaluación y con cross-fitting**, no con el modelo que eligió.



## 5. Baselines y test de diferencia (Prop. 21.1)
Comparamos con tres baselines: **tratar a todos**, **azar al 30 %** (valor esperado $=0{,}3\cdot V(\text{todos})+0{,}7\cdot V(\text{nadie})$) y **no tratar**. El test de diferencia de dos políticas usa las puntuaciones de influencia sobre las **mismas** observaciones:
$$z=\frac{\hat V(\pi_A)-\hat V(\pi_B)}{\widehat{\mathrm{SE}}(\psi_A-\psi_B)}.$$

In [ ]:
_, _, psi_all = todo_pol(np.ones(len(YH), int)); psi_rand = 0.30 * psi_all + 0.70 * psi0                  # azar al 30 %
_, _, psi_top = todo_pol(pi_top); _, _, psi_tree = todo_pol(pi_tree)
def compara(psi_a, psi_b):
    d = psi_a - psi_b; z = d.mean() / (d.std(ddof=1) / np.sqrt(len(d))); return d.mean(), z
filas = {}
for nom, (a, b) in {"top-30 % frente a azar 30 %": (psi_top, psi_rand), "top-30 % frente a tratar a todos (esta viola el presupuesto)": (psi_top, psi_all), "árbol frente a top-30 %": (psi_tree, psi_top), "tratar a todos frente a nadie": (psi_all, psi0)}.items():
    dif, z = compara(a, b); filas[nom] = {"diferencia de valor": dif, "z": z, "|z| > 1,96": "sí" if abs(z) > 1.96 else "no (→ alertar)"}
show(pd.DataFrame(filas).T.round(3))

**Mira el resultado.** Cuando una diferencia entre políticas no es significativa, ¿qué decisión corresponde en el vocabulario del curso?

**Respuesta:**

**Alertar**: la evidencia no permite distinguir esas políticas con la muestra de evaluación. No significa que sean iguales (falta de evidencia ≠ evidencia de igualdad), sino que más sofisticación del targeting no está justificada hoy. Cuando la curva del targeting no se separa de la del azar, la conclusión correcta es que no hay heterogeneidad accionable, y se trata a todos o a nadie según el signo del ATE. (Ojo: «tratar a todos» supera aquí al top-30 % porque todos los efectos son positivos, pero **viola el presupuesto**: la comparación justa con presupuesto es contra el azar al 30 %.)



## 6. «Tres veces el azar»: depende del DGP
El libro afirma que con 30 % de presupuesto el targeting rinde ≈ 3 veces el azar. El cociente es $E[\tau\mid\text{top }30\%]/E[\tau]$, y depende de **cuánta heterogeneidad** hay frente al efecto medio. Con $\tau\sim N(1,s^2)$:

In [ ]:
r = np.random.default_rng(0); filas = {}
for s in (0.25, 0.5, 1.0, 1.7, 3.0):
    tau_ = r.normal(1.0, s, 200000); top = tau_ >= np.quantile(tau_, .70); filas[s] = {"E[τ | top 30 %]": tau_[top].mean(), "cociente frente al azar": tau_[top].mean() / tau_.mean()}
show(pd.DataFrame(filas).T.rename_axis("s (heterogeneidad)").round(2))

**Mira el resultado.** ¿Para qué nivel de heterogeneidad se obtiene «3 veces el azar»? ¿Qué implica para IHDP?

**Respuesta:**

Para $s\approx1{,}7$ con efecto medio 1. Con poca heterogeneidad el cociente es cercano a 1 (el targeting no aporta); con mucha, crece. La cifra del libro es **propiedad de su DGP**, no una constante. En IHDP real la heterogeneidad es moderada, y con una muestra de evaluación pequeña no se podrá distinguir bien.



## 7. Supuesto y bitácora

In [ ]:
inc = psi_top - psi0
dif, z = compara(psi_top, psi_rand)
fila_b = bitacora(
    pregunta="¿Mejora el targeting top-30 % por CATE al azar y a tratar a todos, respetando el presupuesto?",
    metodo="DR-learner para el CATE; política top-30 % y árbol de profundidad 3; valor doblemente robusto en hold-out (validación + prueba)",
    estimando="Ganancia de valor de política frente a no tratar",
    numero_ic=fmt_ci(inc.mean(), inc.std(ddof=1) / np.sqrt(len(inc))) + f"; frente al azar: diferencia {dif:.3f} (z = {z:.2f})",
    supuesto=supuesto("observable", "medido", f"el presupuesto se cumple: el top-30 % trata al {100 * pi_top.mean():.0f} % del hold-out y el árbol al {100 * pi_tree.mean():.0f} %"),
    decision="continuar" if abs(z) > 1.96 else "alertar", lab="L22")

---
## Qué queda para el libro
Quedan en el libro: `econml.policy.PolicyTree`/`PolicyForest` (aquí un árbol de regresión sobre el CATE hace de árbol de política), el teorema de EWM y el ejercicio sobre `nsw_mixtape` (no está en el benchmark).

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*